# Managed Agents: The Agents API

Create a reusable agent, run a task in an OpenAI-hosted environment, continue its session, and download its output files.

[Open in Google Colab](https://colab.research.google.com/github/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/managed_agents_api.ipynb) · [Official quickstart](https://developers.openai.com/api/docs/guides/agents-api/quickstart)

**Harness as a service:** OpenAI manages the agent loop, durable sessions, context compaction, and recovery. We provide the task and choose the tools and environment.

| Concept | What it holds |
|---|---|
| Saved agent | Reusable model and instructions |
| Session | This task's conversation and saved work |
| Turn | One cycle of work within that session |
| Environment | The sandbox where commands and file edits run |

**Filming path (about 4 minutes):** pre-run setup, explain agent versus session, show the creation call, follow streamed progress, then send a follow-up using the same session ID. The stream helper and download code are supplied.

The current Python interface is `client.beta.agents`, distinct from the local **Agents SDK** (`Agent` / `Runner`). You need Agents API access and an application key with `api.agents.read`, `api.agents.write`, and `api.responses.write`. The SDK supplies the beta header. Model and hosted-environment usage incur API charges. This lesson uses synthetic sales data and no external tools or UI automation.

In [ ]:
%pip install openai --upgrade --quiet

In [ ]:
RUN_DEMO = True
RUN_FOLLOW_UP = True
CLEANUP = True  # Download outputs first, then remove only this demo's resources.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from openai import OpenAI
from IPython.display import display, Markdown, FileLink, Image

if not isinstance(RUN_DEMO, bool):
    raise TypeError("RUN_DEMO must be True or False, not text.")
if RUN_DEMO and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(timeout=180, max_retries=1) if RUN_DEMO else None
MODEL = os.getenv("OPENAI_AGENTS_MODEL", "gpt-6-astra")
print("Model:", MODEL)

## 1. Define a reusable agent

The saved agent is a configuration, not an active conversation. We give it a narrow role and ask it to report actual execution results. The session will choose an **OpenAI-hosted environment**, so we do not have to build a shell executor or our own tool loop.

In [ ]:
agent = None
session_id = None
if RUN_DEMO:
    if not hasattr(client.beta, "agents"):
        raise RuntimeError("Update the openai package, then restart the runtime and rerun setup.")
    agent = client.beta.agents.create(
        model=MODEL,
        instructions=("You are a careful data analyst. Use Python's standard library for calculations. "
                      "Run the code and check the actual output. Save requested artifacts under "
                      "/workspace/outputs. Do not use network access or delegate work. "
                      "Use plain text without em dashes."),
    )
    print("Saved agent:", agent.id)

## 2. A small helper to follow progress

Events show work as it happens. We print the model's text and stop on the **root turn's completion**, failure, or cancellation. An `idle` event alone is not proof of success. The helper also remembers the session ID so we can continue the same conversation.

If a stream disconnects, retrieve the saved session and items before retrying. Starting a new session would start a different conversation.

In [ ]:
import time

def consume_turn(events):
    global session_id
    deadline = time.monotonic() + 180
    for event in events:
        if time.monotonic() > deadline:
            raise TimeoutError("Demo time limit reached. Inspect the saved session before retrying.")
        payload = event.model_dump()
        kind = payload["type"]
        if kind == "agent.session.created":
            session_id = payload["session"]["id"]
            print("Session:", session_id)
        else:
            session_id = payload.get("session_id", session_id)
        if kind == "agent.session.turn.output_text.delta":
            print(payload["delta"], end="", flush=True)
        if kind in {"error", "agent.session.failed", "agent.session.environment.failed"}:
            raise RuntimeError(f"Agent lifecycle error: {payload}")
        if kind in {"agent.session.requires_action", "agent.session.environment.requires_action"}:
            raise RuntimeError("This demo needs an unexpected action. Inspect the session.required_actions.")
        turn = payload.get("turn") or {}
        if turn.get("subagent_id") is None:
            if kind in {"agent.session.turn.failed", "agent.session.turn.cancelled"}:
                raise RuntimeError(f"Turn did not complete: {payload}")
            if kind == "agent.session.turn.completed":
                print("\nTurn completed:", payload["turn_id"])
                return payload["turn_id"]
    raise RuntimeError("Stream ended before completion. Retrieve the session and saved items.")

## 3. Start the session and let the managed harness work

The task requires several steps: create code, run it, and write two result files. The managed harness handles that loop. Our application sends the task and consumes events.

We keep the returned **session ID** and **completed turn ID**. A completed turn means the agent finished working, so we still inspect the actual artifacts in the next cells.

In [ ]:
first_turn_id = latest_turn_id = None
if RUN_DEMO:
    try:
        with client.beta.agents.sessions.create(
            agent_id=agent.id,
            environment={"type": "openai_hosted"},
            input=("Create and run a Python script that totals this synthetic revenue: "
                   "North: 120, 140, 160. South: 100, 120, 140. "
                   "Write /workspace/outputs/regional_totals.csv with exactly the columns "
                   "region,total_revenue and one row per region. "
                   "Write /workspace/outputs/weekly_report.txt with regional totals and the grand total. "
                   "Show the script's actual output and check the arithmetic. Keep the task brief."),
            stream=True,
        ) as events:
            first_turn_id = latest_turn_id = consume_turn(events)
    except Exception:
        # These IDs belong only to the resources created by this demo.
        if session_id:
            client.beta.agents.sessions.delete(session_id)
            session_id = None
        if agent:
            client.beta.agents.delete(agent.id)
            agent = None
        raise

## 4. Continue the same session

We open the event stream **before sending the next message** so we do not miss early events. The same session continues with its conversation and existing files. This is a follow-up turn, not a new agent.

Set `RUN_FOLLOW_UP = False` in the parameters cell if you want the shortest first demonstration.

In [ ]:
if RUN_DEMO and RUN_FOLLOW_UP:
    with client.beta.agents.sessions.events.stream(session_id) as events:
        client.beta.agents.sessions.events.create(
            session_id,
            events=[{"type": "agent.session.input.message", "input": [{
                "role": "user", "content": [{"type": "input_text", "text": (
                    "Continue from the existing files. Add one line naming the region with the "
                    "highest revenue to weekly_report.txt. Rerun the calculation and save both "
                    "weekly_report.txt and regional_totals.csv under /workspace/outputs. "
                    "Report the actual results briefly.")}],
            }]}],
        )
        latest_turn_id = consume_turn(events)

## 5. Download published artifacts and check the result

Files under `/workspace/outputs` are published as immutable artifacts when an OpenAI-hosted turn completes. We select the newest published artifact for each **path** and show its **turn ID**. An unchanged file can retain its earlier published version, so it need not have the follow-up turn's ID.

In [ ]:
import csv

workdir = Path("managed_agents_demo")
workdir.mkdir(exist_ok=True)
if RUN_DEMO:
    artifacts = list(client.beta.agents.sessions.artifacts.list(session_id, order="desc"))
    for filename in ("weekly_report.txt", "regional_totals.csv"):
        path = "/workspace/outputs/" + filename
        match = next((a for a in artifacts if a.path == path), None)
        if match is None:
            raise FileNotFoundError(f"Missing published artifact: {path}")
        print(filename, "published by turn:", match.turn_id)
        destination = workdir / filename
        with client.beta.agents.sessions.artifacts.with_streaming_response.content(
            match.id, session_id=session_id,
        ) as content:
            content.stream_to_file(destination)
        display(FileLink(str(destination)))
    print((workdir / "weekly_report.txt").read_text())
    with (workdir / "regional_totals.csv").open() as f:
        actual = {row["region"]: float(row["total_revenue"]) for row in csv.DictReader(f)}
    assert actual == {"North": 420, "South": 360}, f"Calculation mismatch: {actual}"
    print("Verified totals:", actual, "Grand total:", sum(actual.values()))

## 6. Clean up after saving the outputs

Delete this demo's session and saved agent. The downloaded files remain locally. Set `CLEANUP = False` if you want to keep the session and try another follow-up, then run this cell when you are finished.

In [ ]:
if RUN_DEMO and CLEANUP:
    if session_id:
        client.beta.agents.sessions.delete(session_id)
        session_id = None
    if agent:
        client.beta.agents.delete(agent.id)
        agent = None
    print("Deleted this demo's session and saved agent; local artifacts remain.")

## Try it yourself

Keep a session, then ask for a percentage breakdown in a second turn. Inspect the new artifact version and check that percentages add up to 100%. Explore passing input files through `environment.files` in the linked guide.

## Summary

- A saved agent holds reusable configuration; a session holds one continuing conversation and its work.
- The Agents API supplies a managed Codex harness for orchestration, compaction, and recovery.
- Choose an OpenAI-hosted sandbox to run code and edit files without implementing the executor yourself.
- Stream progress, reuse the session ID for follow-ups, and verify downloaded artifacts.
- This hosted API is distinct from using the Agents SDK in your own Python process.

**Troubleshooting:** a 403 may mean your project lacks Agents API access or the required permissions. Update the SDK if `beta.agents` is missing. On a disconnected stream, retrieve `client.beta.agents.sessions.retrieve(session_id)` and list saved `sessions.items` before deciding whether to continue. If a later cell fails, inspect the result and run cleanup once you have saved what you need.

Sources: [Quickstart](https://developers.openai.com/api/docs/guides/agents-api/quickstart), [Agent configuration](https://developers.openai.com/api/docs/guides/agents-api/configuration), [Sessions](https://developers.openai.com/api/docs/guides/agents-api/sessions), [Events](https://developers.openai.com/api/docs/guides/agents-api/sessions/events), [Files and artifacts](https://developers.openai.com/api/docs/guides/agents-api/environments/files).